In [ ]:
# ============================================================
# CUSTOMER CHURN PREDICTION USING MACHINE LEARNING

# ============================================================


# ============================================================
# STEP 1 - IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

import warnings
warnings.filterwarnings('ignore')


# ============================================================
# STEP 2 - LOAD DATASET
# ============================================================

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("================================================")
print("DATASET LOADED SUCCESSFULLY")
print("================================================")

print(df.head())


# ============================================================
# STEP 3 - DATASET INFORMATION
# ============================================================

print("\n================================================")
print("DATASET INFORMATION")
print("================================================")

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Columns:")
print(df.columns)

print("\nDataset Info:")
print(df.info())

print("\nStatistical Summary:")
print(df.describe())


# ============================================================
# STEP 4 - CHECK MISSING VALUES
# ============================================================

print("\n================================================")
print("CHECKING MISSING VALUES")
print("================================================")

print(df.isnull().sum())


# ============================================================
# STEP 5 - DATA CLEANING
# ============================================================

# Convert TotalCharges into numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors='coerce'
)

# Fill missing values
df["TotalCharges"].fillna(
    df["TotalCharges"].median(),
    inplace=True
)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove customerID column
df.drop("customerID", axis=1, inplace=True)

print("\n================================================")
print("DATA CLEANING COMPLETED")
print("================================================")


# ============================================================
# STEP 6 - EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

# ------------------------------------------------
# CHURN DISTRIBUTION
# ------------------------------------------------

plt.figure(figsize=(6,5))

sns.countplot(
    x='Churn',
    data=df
)

plt.title("Customer Churn Distribution")

plt.show()


# ------------------------------------------------
# PIE CHART OF CHURN
# ------------------------------------------------

df['Churn'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    figsize=(6,6)
)

plt.title("Customer Churn Percentage")

plt.ylabel("")

plt.show()


# ------------------------------------------------
# GENDER VS CHURN
# ------------------------------------------------

plt.figure(figsize=(6,5))

sns.countplot(
    x='gender',
    hue='Churn',
    data=df
)

plt.title("Gender vs Churn")

plt.show()


# ------------------------------------------------
# CONTRACT TYPE VS CHURN
# ------------------------------------------------

plt.figure(figsize=(8,5))

sns.countplot(
    x='Contract',
    hue='Churn',
    data=df
)

plt.title("Contract Type vs Churn")

plt.xticks(rotation=15)

plt.show()


# ------------------------------------------------
# INTERNET SERVICE VS CHURN
# ------------------------------------------------

plt.figure(figsize=(8,5))

sns.countplot(
    x='InternetService',
    hue='Churn',
    data=df
)

plt.title("Internet Service vs Churn")

plt.xticks(rotation=15)

plt.show()


# ------------------------------------------------
# MONTHLY CHARGES DISTRIBUTION
# ------------------------------------------------

plt.figure(figsize=(8,5))

sns.histplot(
    df['MonthlyCharges'],
    bins=30,
    kde=True
)

plt.title("Monthly Charges Distribution")

plt.show()


# ------------------------------------------------
# TENURE DISTRIBUTION
# ------------------------------------------------

plt.figure(figsize=(8,5))

sns.histplot(
    df['tenure'],
    bins=30,
    kde=True
)

plt.title("Tenure Distribution")

plt.show()


# ============================================================
# STEP 7 - ENCODE CATEGORICAL VARIABLES
# ============================================================

df = pd.get_dummies(
    df,
    drop_first=True
)

# Fill any remaining missing values
df = df.fillna(0)

print("\n================================================")
print("CATEGORICAL VARIABLES ENCODED")
print("================================================")

print(df.head())


# ============================================================
# STEP 8 - CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(14,10))

sns.heatmap(
    df.corr(),
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")

plt.show()


# ============================================================
# STEP 9 - FEATURE SELECTION
# ============================================================

X = df.drop("Churn_Yes", axis=1)

y = df["Churn_Yes"]

print("\n================================================")
print("FEATURES AND TARGET VARIABLE CREATED")
print("================================================")


# ============================================================
# STEP 10 - FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

print("\n================================================")
print("FEATURE SCALING COMPLETED")
print("================================================")


# ============================================================
# STEP 11 - TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n================================================")
print("TRAIN TEST SPLIT COMPLETED")
print("================================================")


# ============================================================
# STEP 12 - LOGISTIC REGRESSION MODEL
# ============================================================

lr_model = LogisticRegression(
    max_iter=1000
)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("\nLOGISTIC REGRESSION MODEL TRAINED")


# ============================================================
# STEP 13 - DECISION TREE MODEL
# ============================================================

dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("DECISION TREE MODEL TRAINED")


# ============================================================
# STEP 14 - RANDOM FOREST MODEL
# ============================================================

# Hyperparameter Tuning

param_grid = {

    'n_estimators': [100],

    'max_depth': [5, 10],

    'min_samples_split': [2, 5]

}

grid_search = GridSearchCV(

    RandomForestClassifier(random_state=42),

    param_grid,

    cv=3,

    scoring='accuracy'

)

grid_search.fit(X_train, y_train)

rf_model = grid_search.best_estimator_

y_pred_rf = rf_model.predict(X_test)

print("RANDOM FOREST MODEL TRAINED")

print("\nBest Parameters:")
print(grid_search.best_params_)


# ============================================================
# STEP 15 - MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name, y_test, y_pred):

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred)

    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    print("\n================================================")
    print(model_name)
    print("================================================")

    print(f"Accuracy  : {accuracy:.4f}")

    print(f"Precision : {precision:.4f}")

    print(f"Recall    : {recall:.4f}")

    print(f"F1 Score  : {f1:.4f}")

    print("\nClassification Report:")

    print(classification_report(y_test, y_pred))

    # ------------------------------------------------
    # CONFUSION MATRIX
    # ------------------------------------------------

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(f"{model_name} - Confusion Matrix")

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.show()


# ============================================================
# STEP 16 - EVALUATE ALL MODELS
# ============================================================

evaluate_model(
    "LOGISTIC REGRESSION",
    y_test,
    y_pred_lr
)

evaluate_model(
    "DECISION TREE",
    y_test,
    y_pred_dt
)

evaluate_model(
    "RANDOM FOREST",
    y_test,
    y_pred_rf
)


# ============================================================
# STEP 17 - ROC AUC SCORES
# ============================================================

roc_lr = roc_auc_score(
    y_test,
    y_pred_lr
)

roc_dt = roc_auc_score(
    y_test,
    y_pred_dt
)

roc_rf = roc_auc_score(
    y_test,
    y_pred_rf
)

print("\n================================================")
print("ROC-AUC SCORES")
print("================================================")

print("Logistic Regression :", roc_lr)

print("Decision Tree       :", roc_dt)

print("Random Forest       :", roc_rf)


# ============================================================
# STEP 18 - ROC CURVE
# ============================================================

rf_probs = rf_model.predict_proba(X_test)[:,1]

fpr, tpr, thresholds = roc_curve(
    y_test,
    rf_probs
)

plt.figure(figsize=(6,5))

plt.plot(
    fpr,
    tpr,
    label='Random Forest'
)

plt.plot(
    [0,1],
    [0,1],
    linestyle='--'
)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()


# ============================================================
# STEP 19 - CROSS VALIDATION
# ============================================================

cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=5
)

print("\n================================================")
print("CROSS VALIDATION")
print("================================================")

print("Cross Validation Scores:")

print(cv_scores)

print("\nAverage CV Score:")

print(cv_scores.mean())


# ============================================================
# STEP 20 - MODEL COMPARISON
# ============================================================

models = [
    'Logistic Regression',
    'Decision Tree',
    'Random Forest'
]

accuracies = [

    accuracy_score(y_test, y_pred_lr),

    accuracy_score(y_test, y_pred_dt),

    accuracy_score(y_test, y_pred_rf)

]

comparison_df = pd.DataFrame({

    'Model': models,

    'Accuracy': accuracies

})

print("\n================================================")
print("MODEL COMPARISON")
print("================================================")

print(comparison_df)


# ============================================================
# STEP 21 - ACCURACY COMPARISON GRAPH
# ============================================================

plt.figure(figsize=(8,5))

sns.barplot(
    x='Model',
    y='Accuracy',
    data=comparison_df
)

plt.title("Model Accuracy Comparison")

plt.ylim(0,1)

for index, value in enumerate(accuracies):

    plt.text(
        index,
        value + 0.01,
        str(round(value,3))
    )

plt.show()


# ============================================================
# STEP 22 - FEATURE IMPORTANCE
# ============================================================

importance = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({

    'Feature': X.columns,

    'Importance': importance

})

feature_importance_df = feature_importance_df.sort_values(
    by='Importance',
    ascending=False
)

print("\n================================================")
print("TOP IMPORTANT FEATURES")
print("================================================")

print(feature_importance_df.head(10))


# ------------------------------------------------
# FEATURE IMPORTANCE GRAPH
# ------------------------------------------------

plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance_df.head(10)
)

plt.title("Top 10 Important Features")

plt.show()


# ============================================================
# STEP 23 - SAVE RESULTS
# ============================================================

comparison_df.to_csv(
    "Model_Comparison_Results.csv",
    index=False
)

feature_importance_df.to_csv(
    "Feature_Importance.csv",
    index=False
)

print("\n================================================")
print("RESULTS SAVED SUCCESSFULLY")
print("================================================")


# ============================================================
# STEP 24 - FINAL RESULT
# ============================================================

best_model_accuracy = max(accuracies)

best_model = models[
    accuracies.index(best_model_accuracy)
]

print("\n================================================")
print("FINAL RESULT")
print("================================================")

print("BEST MODEL :", best_model)

print(
    "BEST ACCURACY :",
    round(best_model_accuracy,4)
)

print("================================================")


# ============================================================
# END OF PROJECT
# ============================================================